# Predicting arrears next month
Follows `protocol.yaml`: temporal holdout (`holdout.csv`), walk-forward CV on training rows, selection by mean CV ROC AUC, one scoring on holdout.

**Feature decision:** `final_notice_sent` is recorded in the billing cycle *after* the billing month, i.e. alongside/after the outcome, so it is not known at prediction time (it is also near-deterministic with the target). It is excluded, along with `meter_ref` and `billing_month` (per protocol).

In [ ]:
import json, hashlib, numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='42accdc3f8c6ae3514896d73516d98c54e4a27c5422d4fe88e5c4fe1b00a14f7'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='903bb064736772428ae27c90c6c03ac7f9c28c7eb54a50c70b7601b731258ba6'
df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==20939 and len(hold_idx)==7536
TARGET = 'arrears_next_month'
CAT = ['tariff']; NUM = ['occupants','kwh','bill_eur','late_payments_12m']
FEATURES = CAT + NUM
is_hold = np.zeros(len(df), bool); is_hold[hold_idx] = True
train = df[~is_hold]; hold = df[is_hold]
print(len(train), len(hold), train.billing_month.max(), hold.billing_month.min())
assert train.billing_month.max() < hold.billing_month.min()
print(train[TARGET].mean(), hold[TARGET].mean())

## Candidates (named before any scoring)

In [ ]:
def prep(scale):
    return ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), CAT),
                              ('n', StandardScaler() if scale else 'passthrough', NUM)])
CANDIDATES = {
 'baseline_prior': Pipeline([('p', prep(False)), ('m', DummyClassifier(strategy='prior'))]),
 'logreg_C0.1': Pipeline([('p', prep(True)), ('m', LogisticRegression(C=0.1, max_iter=1000))]),
 'logreg_C1': Pipeline([('p', prep(True)), ('m', LogisticRegression(C=1, max_iter=1000))]),
 'hgb_depth3': Pipeline([('p', prep(False)), ('m', HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0))]),
 'hgb_depth6': Pipeline([('p', prep(False)), ('m', HistGradientBoostingClassifier(max_depth=6, learning_rate=0.05, max_iter=200, random_state=0))]),
}
list(CANDIDATES)

## Walk-forward CV on training rows
5 splits over the ordered months: fold k trains on month-blocks 0..k and validates on block k+1. Preprocessing lives inside each pipeline, so it is fitted per fold.

In [ ]:
months = np.sort(train.billing_month.unique())
blocks = np.array_split(months, 6)
folds = []
for k in range(5):
    tr = np.flatnonzero(train.billing_month.isin(np.concatenate(blocks[:k+1])).to_numpy())
    va = np.flatnonzero(train.billing_month.isin(blocks[k+1]).to_numpy())
    folds.append((tr, va))
Xtr, ytr = train[FEATURES], train[TARGET].to_numpy()
cv_scores = {}
for name, est in CANDIDATES.items():
    s = []
    for tr, va in folds:
        m = clone(est).fit(Xtr.iloc[tr], ytr[tr])
        s.append(float(roc_auc_score(ytr[va], m.predict_proba(Xtr.iloc[va])[:,1])))
    cv_scores[name] = s
cv = pd.DataFrame(cv_scores).T
cv['mean'] = cv.mean(axis=1)
cv.sort_values('mean', ascending=False)

## Choose on CV only, refit on all training rows, score holdout once

In [ ]:
chosen = max(cv_scores, key=lambda n: np.mean(cv_scores[n]))
print('chosen:', chosen)
final = clone(CANDIDATES[chosen]).fit(Xtr, ytr)
holdout_auc = float(roc_auc_score(hold[TARGET], final.predict_proba(hold[FEATURES])[:,1]))
print('holdout roc_auc:', holdout_auc)

In [ ]:
# row labels: 0-based positions in data.csv
train_pos = np.flatnonzero(~is_hold)
evidence = {
 'features': FEATURES,
 'candidates': list(CANDIDATES),
 'cv_folds': [[train_pos[tr].tolist(), train_pos[va].tolist()] for tr, va in folds],
 'cv_scores': cv_scores,
 'chosen': chosen,
 'holdout_rows': hold_idx.tolist(),
 'holdout_scores': {chosen: {'roc_auc': holdout_auc}},
}
json.dump(evidence, open('evidence.json', 'w'))

## Conclusion
See the CV table and holdout ROC AUC above. Only features known at the end of the billing month are used; `final_notice_sent` was deliberately left out as post-outcome information.